# 06 · Pereira Study vs. This Thesis — Rank Bump Chart

Compares each language's CPU-energy-efficiency rank between the Pereira et al. (2021)
study (\cite{PEREIRA2021102609}, *Ranking Programming Languages by Energy Efficiency*)
and this thesis (2026), 10 years apart.

**Scope:** restricted to the 18 languages this thesis also measured. Pereira's original
ranks (1–27) are filtered down to those 18 and renumbered 1–18, so the two columns are
directly comparable — the excluded languages (Ada, Pascal, Chapel, Lisp, Fortran, Racket,
TypeScript, Hack, JRuby) never appear in this thesis.

**Colour:** lines are coloured **per language**, not by execution model. Pereira's
compiler/interpreter configuration isn't necessarily identical to this thesis's (e.g.
Pereira ran Java on the standard JDK, this thesis uses GraalVM Native Image), so the
AOT/JIT/Interpreted label from `plot_style.COMPILER` describes *this thesis's* setup and
would be misleading applied to Pereira's side.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import matplotlib.pyplot as plt

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

## 1. Rank Data

Both rankings are by mean CPU energy (lower = more efficient = rank 1).

In [ ]:
# Pereira study rank, filtered to the 18 languages this thesis also measured and
# renumbered 1..18 (original ranks skip languages not in our set).
PEREIRA_RANK = {
    "C": 1, "Rust": 2, "C++": 3, "Java": 4, "OCaml": 5, "Swift": 6,
    "Haskell": 7, "C#": 8, "Go": 9, "Dart": 10, "F#": 11, "JavaScript": 12,
    "PHP": 13, "Erlang": 14, "Lua": 15, "Ruby": 16, "Python": 17, "Perl": 18,
}

# This thesis's rank by mean CPU energy (18 languages, all measured).
THESIS_RANK = {
    "C++": 1, "C": 2, "C#": 3, "Rust": 4, "F#": 5, "Java": 6, "Go": 7,
    "OCaml": 8, "JavaScript": 9, "Haskell": 10, "Dart": 11, "Swift": 12,
    "PHP": 13, "Erlang": 14, "Lua": 15, "Ruby": 16, "Python": 17, "Perl": 18,
}

assert set(PEREIRA_RANK) == set(THESIS_RANK), "language sets must match for a fair bump chart"
N_LANGUAGES = len(THESIS_RANK)

## 2. Bump Chart

One line per language, coloured distinctly (hand-picked 18-colour qualitative palette,
assigned alphabetically for a deterministic mapping) so movement between the two
studies is easy to trace by eye.

In [ ]:
# Distinct per-language colour (not compiler-based — see note above). Hand-picked
# instead of tab20 directly: tab20's alternating light/dark pairs include a pale
# gray that is nearly invisible on a white background. Assigned alphabetically for
# a stable, reproducible mapping.
PALETTE_18 = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78",
    "#98df8a", "#ff9896", "#c5b0d5", "#c49c94", "#f7b6d2", "#000000",
]
languages_sorted = sorted(THESIS_RANK)
LANG_COLOR = dict(zip(languages_sorted, PALETTE_18))


def plot_bump_chart() -> plt.Figure:
    """Draw the two-column bump chart of rank changes between studies.

    Behaviour: for each of the 18 common languages, plots a line from its
    Pereira-study rank (left column) to its thesis rank (right column),
    coloured per-language via LANG_COLOR. Labels each endpoint with the
    language name and adds bold rank-number ticks 1..N on the y-axis. Returns
    the Figure; does not save it (caller decides whether to call ps.save_fig).
    """
    fig, ax = plt.subplots(figsize=(8, 10))

    for lang, thesis_r in THESIS_RANK.items():
        pereira_r = PEREIRA_RANK[lang]
        color = LANG_COLOR[lang]
        ax.plot([0, 1], [pereira_r, thesis_r], color=color, marker="o",
                 markersize=7, linewidth=2.2, zorder=3)
        ax.text(-0.04, pereira_r, lang, ha="right", va="center",
                fontweight="bold", fontsize=10, color=color)
        ax.text(1.04, thesis_r, lang, ha="left", va="center",
                fontweight="bold", fontsize=10, color=color)

    ax.set_xlim(-0.35, 1.35)
    ax.set_ylim(N_LANGUAGES + 0.6, 0.4)  # inverted: rank 1 at top
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Pereira Study (2021)", "This Thesis (2026)"],
                        fontsize=11, fontweight="bold")
    ax.set_yticks(range(1, N_LANGUAGES + 1))
    ax.set_yticklabels(range(1, N_LANGUAGES + 1), fontweight="bold")
    ax.set_ylabel("")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", left=False)
    fig.tight_layout()
    return fig


fig = plot_bump_chart()
ps.save_fig(fig, "06_pereira_bump_chart")
plt.show()

> **Takeaway:** the bottom third of the ranking (PHP, Erlang, Lua, Ruby, Python, Perl) is
> unchanged across 10 years — flat lines. Most movement happens among the AOT-compiled
> languages clustered near the top; e.g. C and C++ swap the #1/#3 spots and F# jumps from
> 11th to 5th.